# Generating embeddings

## Pre-processing

In [5]:
import pandas as pd

In [14]:
data = pd.read_csv('../data/petitions.csv')

data.head()

,id,public_petition_text,reason_category
0,3168490,снег на дороге,Благоустройство
1,3219678,очистить кабельный киоск от рекламы,Благоустройство
2,2963920,"Просим убрать все деревья и кустарники, которы...",Благоустройство
3,3374910,Неудовлетворительное состояние парадной - надп...,Содержание МКД
4,3336285,Граффити,Благоустройство


In [15]:
petitions = data['public_petition_text'].dropna().astype(str)

petitions.head()

0                                       снег на дороге
1                  очистить кабельный киоск от рекламы
2    Просим убрать все деревья и кустарники, которы...
3    Неудовлетворительное состояние парадной - надп...
4                                             Граффити
Name: public_petition_text, dtype: str

In [12]:
import re
import nltk
import pymorphy3
from razdel import tokenize
from nltk.corpus import stopwords

nltk.download('stopwords')
russian_stopwords = set(stopwords.words('russian'))

morph = pymorphy3.MorphAnalyzer()

def preprocess_text(text):

    text = text.lower()

    # Удаление URL
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Удаление email
    text = re.sub(r'\S+@\S+', '', text)

    # Замена числительных на специальный токен
    text = re.sub(r'\d+', 'NUM', text)

    # Токенизация через razdel
    tokens = [token.text for token in tokenize(text)]

    # Убирает точки, запятые, скобки, тире, кавычки и смайлики
    clean_tokens = [
        t for t in tokens 
        if re.search(r'[а-яёa-z]', t) or t == 'NUM'
    ]

    # Удаление стоп-слов
    filtered_tokens = [word for word in clean_tokens if word not in russian_stopwords]

    # Лемматизация
    lemmatized_words = [morph.parse(word)[0].normal_form for word in filtered_tokens]

    return lemmatized_words

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/irinaaristova/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# raw_text = "Привет! Заходи на сайт https://test.ru или пиши на mail@ya.ru. Купи 2 диван-кровати за 15000 руб."
# print(preprocess_text(raw_text))

['привет', 'заходить', 'сайт', 'писать', 'купить', 'num', 'диван-кровать', 'num', 'руб']


### Dataset tokenization

In [17]:
from tqdm.auto import tqdm

tqdm.pandas()

processed_tokens = petitions.progress_apply(preprocess_text)

processed_tokens = processed_tokens[processed_tokens.apply(len) > 0].reset_index(drop=True)

processed_tokens.to_pickle('../data/petitions_tokens.pkl')

  0%|          | 0/59889 [00:00<?, ?it/s]

In [ ]:
processed_tokens = pd.read_pickle('../data/petitions_tokens.pkl')

### CBoW and SkipGram

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

word_counts = {}
for text in processed_tokens:
    for word in text:
        word_counts[word] = word_counts.get(word, 0) + 1

word_to_idx = {'<UNK>': 0}

for word, count in word_counts.items():
    if count >= 5:
        word_to_idx[word] = len(word_to_idx)

idx_to_word = {idx: word for word, idx in word_to_idx.items()}

VOCAB_SIZE = len(word_to_idx)

Всего слов в корпусе: 635409
Уникальных токенов до фильтрации: 17804
Размер итогового словаря (VOCAB_SIZE): 6091
Отсеяно редких слов: 11714


In [ ]:
import torch
from torch import nn

context_window = 2 # два слова слева и два справа?
cbow_x, cbow_y, sg_x, sg_y = [], [], [], []

for text in processed_tokens:
    indices = [word_to_idx.get(w, 0) for w in text]
    if len(indices) < context_window * 2 + 1:
        continue
        
    for i in range(context_window, len(indices) - context_window):
        center = indices[i]
        context = indices[i - context_window : i] + indices[i + 1 : i + context_window + 1]
        
        # CBOW: Контекст -> Центральное слово
        cbow_x.append(context)
        cbow_y.append(center)
        
        # Skip-Gram: Центральное слово -> Контекст
        for ctx_word in context:
            sg_x.append(center)
            sg_y.append(ctx_word)

# Конвертация в тензоры
cbow_x = torch.tensor(cbow_x, dtype=torch.long)
cbow_y = torch.tensor(cbow_y, dtype=torch.long)
sg_x = torch.tensor(sg_x, dtype=torch.long)
sg_y = torch.tensor(sg_y, dtype=torch.long)

class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        return self.linear(self.embeddings(x).mean(dim=1))

class SkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        return self.linear(self.embeddings(x))

In [ ]:
def train_model(model, x_tensor, y_tensor, epochs=10, batch_size=256, lr=0.005):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    dataset_size = x_tensor.size(0)
    
    model.train()
    for epoch in range(epochs):
        # Перемешивание индексов для батчей
        indices = torch.randperm(dataset_size)
        total_loss = 0
        
        for i in range(0, dataset_size, batch_size):
            batch_idx = indices[i : i + batch_size]
            batch_x, batch_y = x_tensor[batch_idx], y_tensor[batch_idx]
            
            optimizer.zero_grad()
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss / (dataset_size // batch_size + 1):.4f}")
    return model

EMBED_DIM = 100
cbow_model = train_model(CBOW(VOCAB_SIZE, EMBED_DIM), cbow_x, cbow_y)
sg_model = train_model(SkipGram(VOCAB_SIZE, EMBED_DIM), sg_x, sg_y)

In [ ]:
def extract_and_save_embeddings(model, prefix_name, target_dim=3):
    # Извлечение матрицы весов (эмбеддингов)
    embeddings = model.embeddings.weight.detach().numpy()
    
    # Сжатие размерности через PCA
    pca = PCA(n_components=target_dim)
    embeddings_pca = pca.fit_transform(embeddings)
    
    # Сохранение векторов
    pd.DataFrame(embeddings_pca).to_csv(
        f'{prefix_name}_vectors.tsv', sep='\t', index=False, header=False
    )
    
    # Сохранение метаданных (слов) с сохранением порядка индексов
    words = [idx_to_word[i] for i in range(VOCAB_SIZE)]
    pd.DataFrame(words).to_csv(
        f'{prefix_name}_metadata.tsv', sep='\t', index=False, header=False
    )

# Сохраняем результаты обеих моделей 
extract_and_save_embeddings(cbow_model, 'cbow')     # куда это сохраняется
extract_and_save_embeddings(sg_model, 'skipgram')